# Phase 4: Baseline Models & Benchmark Evaluation

**Course:** Advanced AI Techniques (AAI)  
**Project:** AI-Enabled Micro-channel Battery Thermal Management System for Electric Vehicles (KJS-CES-02)  
**Dataset:** FreeTwinEV P45B 1s4p Battery Module Measurement  
**Status:** Phase 4 Benchmark Evaluation Completed  

This notebook establishes rigorous empirical baselines before considering complex non-linear machine learning models:
1. **Naive Persistence Baseline:** Predicts $\Delta T = 0$ ($T_{t+h} = T_t$).
2. **Mean Baseline:** Predicts constant training-set mean $\Delta T$.
3. **Linear Regression Benchmark:** Ordinary Least Squares on the 9-feature Baseline set.
4. **Ridge Regression Benchmark:** L2-regularized linear model (fixed default $\alpha=1.0$) on scaled features.
5. **Extended Feature Linear Benchmarks:** Evaluating whether the 12 derived Phase 2 features offer immediate linear predictive gains.

> **Strict Protocol Constraint:** All baselines are fitted strictly on the 10 Training runs. Validation runs (ID16, ID22) are evaluated pooled and per-run. The Final Test runs (ID18, ID19) remain **completely untouched**.

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# Configure paths
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root / "src"))

from freetwinev_baselines import (
    NaivePersistenceBaseline,
    MeanBaseline,
    LinearRegressionBaseline,
    RidgeRegressionBaseline,
    compute_metrics,
    audit_baseline_protocol,
    BASELINE_9_FEATURES,
    EXTENDED_21_FEATURES,
    TRAIN_RUNS, VAL_RUNS, TEST_RUNS
)

results_dir = project_root / "results" / "freetwinev"
preprocessed_dir = results_dir / "preprocessed"
print("Preprocessed directory exists:", preprocessed_dir.exists())

## 1. Load Preprocessed Datasets & Verify Schemas

We load the unscaled and scaled CSVs generated in Phase 3 for both 60s and 300s prediction horizons.

In [ ]:
train_60 = pd.read_csv(preprocessed_dir / "baseline_60s_train.csv")
val_60 = pd.read_csv(preprocessed_dir / "baseline_60s_val.csv")

train_300 = pd.read_csv(preprocessed_dir / "baseline_300s_train.csv")
val_300 = pd.read_csv(preprocessed_dir / "baseline_300s_val.csv")

print(f"60s Horizon:  Train rows = {len(train_60)}, Val rows = {len(val_60)}")
print(f"300s Horizon: Train rows = {len(train_300)}, Val rows = {len(val_300)}")
print(f"Features in Baseline 9: {BASELINE_9_FEATURES}")

## 2. Automated Scientific Protocol & Leakage Audit

Before fitting any baseline model, we execute automated assertions verifying:
- Test runs (ID18, ID19) are completely absent from training and validation data.
- No target columns appear in the feature matrix X.
- No future measurements or lead-shifted variables appear in X.
- Training and validation run sets are completely disjoint.

In [ ]:
audit_60 = audit_baseline_protocol(
    train_60, val_60, BASELINE_9_FEATURES, "target_DeltaT_60s", "60s_baseline"
)
audit_300 = audit_baseline_protocol(
    train_300, val_300, BASELINE_9_FEATURES, "target_DeltaT_300s", "300s_baseline"
)

print("60s Audit Overall Status:", audit_60["overall_status"])
print("300s Audit Overall Status:", audit_300["overall_status"])
for k, v in audit_60["checks"].items():
    print(f"  - {k}: {v['status']}")

## 3. Load Persisted Benchmark Results & Tables

We display the benchmark results computed across all models on the validation set.

In [ ]:
pooled_results = pd.read_csv(results_dir / "baseline_results.csv")
per_run_results = pd.read_csv(results_dir / "baseline_results_per_run.csv")

display_cols = ["Horizon", "Model", "Feature_Set", "Val_MAE", "Val_RMSE", "Val_R2", "Val_Bias", "Val_Max_AE"]
print("=== POOLED VALIDATION BENCHMARK RESULTS ===")
print(pooled_results[display_cols].to_string(index=False))

print("\n=== PER-RUN VALIDATION RESULTS (ID16 vs ID22) ===")
per_run_cols = ["Horizon", "Model", "Feature_Set", "Run", "MAE", "RMSE", "R2", "Bias", "Max_AE"]
print(per_run_results[per_run_cols].to_string(index=False))

## 4. Reconstructed Absolute Temperature Evaluation

For each model, the predicted future cell temperature is reconstructed as:
$$T_{\text{pred}}(t+h) = T_{\text{cell,max}}(t) + \hat{\Delta T}$$
Evaluating on absolute temperature reveals how thermal autocorrelation affects raw temperature metrics vs. dynamic temperature-rise metrics.

In [ ]:
abs_cols = ["Horizon", "Model", "Feature_Set", "Val_Abs_MAE", "Val_Abs_RMSE", "Val_Abs_R2"]
print("=== RECONSTRUCTED ABSOLUTE TEMPERATURE BENCHMARK ===")
print(pooled_results[abs_cols].to_string(index=False))

## 5. Diagnostic Figures & Error Visualizations

We display the 5 publication-grade diagnostic figures generated in Phase 4.

In [ ]:
from IPython.display import Image, display
figs = [
    "11_baseline_actual_vs_predicted.png",
    "12_baseline_residual_distributions.png",
    "13_baseline_residuals_vs_target.png",
    "14_baseline_absolute_temp_tracking.png",
    "15_baseline_per_run_comparison.png"
]
for f in figs:
    f_path = results_dir / "figures" / f
    if f_path.exists():
        print(f"\nFigure: {f}")
        display(Image(filename=str(f_path)))

## 6. Phase 4 Summary & Findings

1. **Persistence is Defeated:** Linear Regression on 9 Baseline features reduces 60s pooled validation RMSE from $1.574\text{ }^\circ\text{C}$ to $1.013\text{ }^\circ\text{C}$ ($R^2 = 0.5858$), showing genuine physical predictive power on real experimental data.
2. **Derived Features Yield Major Linear Gains:** Adding the 12 Phase 2 features (Extended 21) reduces 60s validation MAE to $0.160\text{ }^\circ\text{C}$ and RMSE to $0.723\text{ }^\circ\text{C}$ ($R^2 = 0.7890$).
3. **Chilled Run (ID22) Exceptionally Well Modeled:** Linear Regression on Extended features achieves $R^2 = 0.9101$ with MAE of $0.116\text{ }^\circ\text{C}$ on ID22.
4. **Extreme Load (ID16) Reveals Thermal Non-Linearity:** Under -120A continuous cycling, linear models exhibit residual curvature, motivating Phase 5 non-linear models (Random Forest, Gradient Boosting).
5. **Integrity Maintained:** Final test runs (ID18, ID19) remain completely untouched.